In [6]:
%pip install -U langchain-ollama langchain-community langchain-classic langchain-core langchain-chroma chromadb faiss-cpu wikipedia

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document

# Local embedding model
embedding_model = OllamaEmbeddings(model="nomic-embed-text")

# Local chat model for LLM-powered retrievers
llm = ChatOllama(model="qwen2.5:3b", temperature=0)

print("Ollama models initialized")

Ollama models initialized


Wikipedia Retriever

In [8]:
import wikipedia
from langchain_community.retrievers import WikipediaRetriever

# Wikimedia API requires a custom User-Agent to avoid HTTP 403 / JSONDecodeError
wikipedia.set_user_agent("MyLangChainApp/1.0 (contact@example.com)")


In [9]:
# Initialize the retriever
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [12]:
import requests
from langchain_community.retrievers import WikipediaRetriever

# Set a proper User-Agent
headers = {
    "User-Agent": "LangChainStudentProject/1.0 (educational project; contact: your-email@example.com)"
}

# Test Wikipedia API
url = "https://en.wikipedia.org/w/api.php"

params = {
    "action": "query",
    "format": "json",
    "titles": "India"
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=15
)

print("Status:", response.status_code)
print(response.json())

Status: 200
{'batchcomplete': '', 'query': {'pages': {'14533': {'pageid': 14533, 'ns': 0, 'title': 'India'}}}}


In [13]:
import requests

def wikipedia_search(query):
    url = "https://en.wikipedia.org/w/api.php"

    headers = {
        "User-Agent": "LangChainStudentProject/1.0 (educational project; contact: your-email@example.com)"
    }

    params = {
        "action": "query",
        "format": "json",
        "generator": "search",
        "gsrsearch": query,
        "gsrlimit": 2,
        "prop": "extracts",
        "exintro": True,
        "explaintext": True
    }

    response = requests.get(
        url, params=params, headers=headers, timeout=20
    )
    response.raise_for_status()

    data = response.json()
    pages = data.get("query", {}).get("pages", {})

    return [
        page["extract"]
        for page in pages.values()
        if page.get("extract")
    ]

docs = wikipedia_search("History of India")

for doc in docs:
    print(doc)

The Indus Valley Civilisation, the early civilisation of India and Pakistan, developed an economy based on agriculture and craft that later spread into central India. Angus Maddison estimates that from 1-1000 AD, the regions making up present-day India contributed roughly 30% of the world's population and GDP. India experienced per-capita GDP growth in the high medieval era.
By the late 17th century, the Mughal Empire under Aurangzeb reached its greatest extent and controlled most of South Asia by the time of Aurangzeb's death in 1707. The Indian economy, Maddison estimates, produced about 27% of global GDP by the early 18th century under Aurangzeb's reign.
Until the 18th century, India was one of the most important manufacturing centres in international trade. This manufacturing growth is often seen as a form of proto-industrialisation, similar to 18th-century Western Europe before the Industrial Revolution. According to Jeffrey Williamson, the Indian subcontinent deindustrialised in 

In [14]:
# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

# Display retrieved documents
for i, doc in enumerate(docs):
    print(f"--- Document {i+1}: {doc.metadata.get('title')} ---")
    print(doc.page_content[:300])
    print()


--- Document 1: India–Pakistan war of 1971 ---
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971. The war began wi

--- Document 2: India–Pakistan war of 1965 ---
The India–Pakistan war of 1965, also known as the second Kashmir war, was an armed conflict between Pakistan and India that took place from August 1965 to September 1965.
The conflict began following Pakistan's unsuccessful Operation Gibraltar, which was designed to infiltrate forces into Jammu and 



Vector Store Retriever (Chroma)

In [15]:
from langchain_chroma import Chroma

In [16]:
# Step 1: Source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="Ollama provides locally running language and embedding models."),
]

In [18]:
# Step 2: Create Chroma vector store using Ollama embeddings
# A separate collection name avoids mixing vectors made by another embedding model.
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection_ollama"
)

In [19]:
# Step 3: Convert vector store into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [20]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [21]:
for i, doc in enumerate(results):
    print(f"\\n--- Result {i+1} ---")
    print(doc.page_content)

\n--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.
\n--- Result 2 ---
Embeddings convert text into high-dimensional vectors.


In [22]:
# Direct similarity search
results = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results):
    print(f"\\n--- Result {i+1} ---")
    print(doc.page_content)

\n--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.
\n--- Result 2 ---
Embeddings convert text into high-dimensional vectors.


MMR (Maximal Marginal Relevance) Retriever

In [23]:
from langchain_community.vectorstores import FAISS

In [24]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [25]:
# Create FAISS vector store with Ollama embeddings
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [26]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5}
)

In [27]:
query = "What is LangChain?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\\n--- Result {i+1} ---")
    print(doc.page_content)

\n--- Result 1 ---
LangChain is used to build LLM based applications.
\n--- Result 2 ---
LangChain supports Chroma, FAISS, Pinecone, and more.
\n--- Result 3 ---
Embeddings are vector representations of text.


MultiQuery Retriever

In [28]:
from langchain_classic.retrievers import MultiQueryRetriever

In [29]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [30]:
# Create FAISS vector store with Ollama embeddings
vectorstore = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

In [31]:
# Ordinary similarity retriever
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# MultiQueryRetriever uses the local Ollama chat model to generate alternative queries
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=llm
)

In [32]:
# Query
query = "How to improve energy levels and maintain balance?"

# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [33]:
for i, doc in enumerate(similarity_results):
    print(f"\\n--- Similarity Result {i+1} ---")
    print(doc.page_content)

print("*" * 100)

for i, doc in enumerate(multiquery_results):
    print(f"\\n--- MultiQuery Result {i+1} ---")
    print(doc.page_content)

\n--- Similarity Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
\n--- Similarity Result 2 ---
The solar energy system in modern homes helps balance electricity demand.
\n--- Similarity Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
\n--- Similarity Result 4 ---
Photosynthesis enables plants to produce energy by converting sunlight.
\n--- Similarity Result 5 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.
****************************************************************************************************
\n--- MultiQuery Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
\n--- MultiQuery Result 2 ---
The solar energy system in modern homes helps balance electricity demand.
\n--- MultiQuery Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.
\n--- MultiQuery Result 4 ---
Mindful

ContextualCompressionRetriever

In [34]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [35]:
# Recreate the document objects
docs = [
    Document(
        page_content=(
            "The Grand Canyon is one of the most visited natural wonders in the world. "
            "Photosynthesis is the process by which green plants convert sunlight into energy. "
            "Millions of tourists travel to see it every year. The rocks date back millions of years."
        ),
        metadata={"source": "Doc1"}
    ),
    Document(
        page_content=(
            "In medieval Europe, castles were built primarily for defense. "
            "The chlorophyll in plant cells captures sunlight during photosynthesis. "
            "Knights wore armor made of metal. Siege weapons were often used to breach castle walls."
        ),
        metadata={"source": "Doc2"}
    ),
    Document(
        page_content=(
            "Basketball was invented by Dr. James Naismith in the late 19th century. "
            "It was originally played with a soccer ball and peach baskets. NBA is now a global league."
        ),
        metadata={"source": "Doc3"}
    ),
    Document(
        page_content=(
            "The history of cinema began in the late 1800s. Silent films were the earliest form. "
            "Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells. "
            "Modern filmmaking involves complex CGI and sound design."
        ),
        metadata={"source": "Doc4"}
    )
]

In [36]:
# Create FAISS vector store with Ollama embeddings
vectorstore = FAISS.from_documents(docs, embedding_model)

# Base retriever
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [37]:
# Set up the compressor using the local Ollama chat model
compressor = LLMChainExtractor.from_llm(llm)

In [39]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [40]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

for i, doc in enumerate(compressed_results):
    print(f"\\n--- Result {i+1} ---")
    print(doc.page_content)